In [88]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [89]:
load_dotenv()
llm= ChatGroq(model='openai/gpt-oss-120b')

In [90]:
class substate():
    input:str
    translate_text:str


In [91]:

def translate_text(state:substate)->substate:
    user=state['input']
    prompt=f"""Translate the following text to urdu.
Keep it natural and clear. Do not add extra content.

Text:
{user}"""
    
    response=llm.invoke(prompt)

    return {
        'translate_text':response.content
    }

In [92]:
graph=StateGraph(substate)

#nodes
graph.add_node('translate_text', translate_text)
#edges
graph.add_edge(START, 'translate_text')
graph.add_edge('translate_text', END)

subgraph =graph.compile()

parent graph

In [93]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_urdu: str

In [94]:
def gen_node(state:ParentState)->ParentState:
    question=state['question']
    prompt=f"""you are helpful assistant, give me answer clearly of the {question}"""
    response= llm.invoke(prompt)
    return{
        'answer_eng':response
    }


    # answer_eng lo
    # subgraph ko bhejo  through invoke lekn isko subgraph ko bejana hai and state ko fill karna hai 
    # urdu result lo
    # answer_urdu mein save karo
def translate(state:ParentState)->ParentState:
    # call the subgraph
    eng_answer=state['answer_eng']
    result = subgraph.invoke({'input':eng_answer}) #input test ye subgraph state me hai

    return {'answer_urdu': result['translate_text']}


In [95]:
parent_graph= StateGraph(ParentState)

#nodes
parent_graph.add_node('gen_node', gen_node)
parent_graph.add_node('translate', translate)

# edges
parent_graph.add_edge(START, 'gen_node')
parent_graph.add_edge('gen_node', 'translate')
parent_graph.add_edge('translate', END)

graph=parent_graph.compile()


In [96]:
graph.invoke({'question': 'What is quantum physics'})

{'question': 'What is quantum physics',
 'answer_eng': AIMessage(content='**Quantum physics (or quantum mechanics) is the branch of science that describes how the tiniest pieces of the universe—atoms, electrons, photons, and other sub‑atomic particles—behave.**  \n\n### Why it’s needed\nClassical physics (Newton’s laws, Maxwell’s electromagnetism, etc.) works extremely well for everyday sizes and speeds, but it fails when we look at:\n- **Very small scales** (≈\u202f10⁻⁹\u202fm and below)\n- **Very low temperatures** (where thermal motion is minimal)\n- **Very high energies** (close to the speed of light)\n\nAt these regimes, nature follows a different set of rules—those of quantum physics.\n\n---\n\n## Core Ideas (in plain language)\n\n| Concept | What it means | Everyday analogy (rough) |\n|---------|---------------|--------------------------|\n| **Quantization** | Energy, angular momentum, and other properties come in discrete “chunks” (quanta) rather than a smooth continuum. | Like